In [1]:
%pip install numpy
%pip install treys
%pip install gym
%pip install pokerenv

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pokerenv.obs_indices as indices
from pokerenv.table import Table
from pokerenv.common import PlayerAction, Action, action_list


class ExampleRandomAgent:
    def __init__(self):
        self.actions = []
        self.observations = []
        self.rewards = []

    def get_action(self, observation):
        self.observations.append(observation)
        valid_actions = np.argwhere(observation[indices.VALID_ACTIONS] == 1).flatten()
        valid_bet_low = observation[indices.VALID_BET_LOW]
        valid_bet_high = observation[indices.VALID_BET_HIGH]
        chosen_action = PlayerAction(np.random.choice(valid_actions))
        bet_size = 0
        if chosen_action is PlayerAction.BET:
            bet_size = np.random.uniform(valid_bet_low, valid_bet_high)
        table_action = Action(chosen_action, bet_size)
        print()
        self.actions.append(table_action)
        return table_action

    def reset(self):
        self.actions = []
        self.observations = []
        self.rewards = []

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [3]:
active_players = 6
agents = [ExampleRandomAgent() for _ in range(6)]
player_names = {
    0: "TrackedAgent1",
    1: "Agent2",
}  # Rest are defaulted to player3, player4...
# Should we only log the 0th players (here TrackedAgent1) private cards to hand history files
track_single_player = True
# Bounds for randomizing player stack sizes in reset()
low_stack_bbs = 50
high_stack_bbs = 200
hand_history_location = "hands/"
invalid_action_penalty = 0
table = Table(
    active_players,
    player_names=player_names,
    track_single_player=track_single_player,
    stack_low=low_stack_bbs,
    stack_high=high_stack_bbs,
    hand_history_location=hand_history_location,
    invalid_action_penalty=invalid_action_penalty,
)
table.seed(1)

In [4]:
iteration = 1
while True:
    if iteration % 50 == 0:
        table.hand_history_enabled = True
    active_players = np.random.randint(2, 7)
    table.n_players = active_players
    obs = table.reset()
    for agent in agents:
        agent.reset()
    acting_player = int(obs[indices.ACTING_PLAYER])
    try:
        while True:
            action = agents[acting_player].get_action(obs)
            try:   
                obs, reward, done, _ = table.step(action)
            except Exception:
                print(obs)
                print(reward)
                print(done)
                raise Exception
                
            if done:
                # Distribute final rewards
                for i in range(active_players):
                    agents[i].rewards.append(reward[i])
                break
            else:
                # This step can be skipped unless invalid action penalty is enabled,
                # since we only get a reward when the pot is distributed, and the done flag is set
                agents[acting_player].rewards.append(reward[acting_player])
                acting_player = int(obs[indices.ACTING_PLAYER])
    except:
        break        
    iteration += 1
    table.hand_history_enabled = False










[  5.     1.     0.     1.     0.     1.    12.09   3.     4.     4.
   1.     9.    12.09  92.91   0.     1.     4.     0.     4.     8.
 338.73   0.     0.     0.     1.    36.09  92.91   0.     0.     1.
   0.   179.     1.     0.     0.     2.     1.    25.09  92.91   0.
   0.     4.     1.     0.    59.     0.     1.     5.     0.   118.
   0.     0.     0.     0.     0.     0.     0.     0.  ]
[0 0 0 0 0 0]
False
